# Xente Transaction EDA — Credit Risk Proxy

Exploratory analysis for RFM-based proxy target and feature engineering.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW_PATH = Path("../data/raw/training.csv")
if not RAW_PATH.exists():
    raise FileNotFoundError("Place training.csv in data/raw/")

df = pd.read_csv(RAW_PATH)
df["TransactionStartTime"] = pd.to_datetime(df["TransactionStartTime"])
df.head()

In [ ]:
print(f"Transactions: {len(df):,}")
print(f"Customers: {df['CustomerId'].nunique():,}")
print(f"Fraud rate: {df['FraudResult'].mean():.4f}")
df.describe(include='all').T.head(15)

In [ ]:
import sys
sys.path.insert(0, "..")
from src.data_processing import build_modeling_dataset, compute_weight_of_evidence

dataset, metadata = build_modeling_dataset(df)
print("Default proxy rate:", metadata["default_rate"])
pd.DataFrame(metadata["iv_summary"]).head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ["recency_days", "frequency", "monetary_total"]):
    dataset[col].hist(ax=ax, bins=30, edgecolor="k")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
corr = dataset[[c for c in metadata["selected_features"] if c in dataset.columns] + ["default_risk"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation with proxy default")
plt.tight_layout()
plt.show()